In [1]:
from dolfinx.io import gmsh
from mpi4py import MPI

out = gmsh.read_from_msh("rectangle.msh", MPI.COMM_WORLD, rank=0, gdim=2)
domain, ct, ft = (out.mesh, out.cell_tags, out.facet_tags) if hasattr(out, "mesh") else out

Info    : Reading 'rectangle.msh'...
Info    : 9 entities
Info    : 994 nodes
Info    : 1986 elements
Info    : Done reading 'rectangle.msh'


In [2]:
import ufl 
tdim = domain.topology.dim 
fdim = tdim - 1
domain.topology.create_connectivity(fdim, tdim) 

ds = ufl.Measure("ds", domain=domain, subdomain_data=ft)
dx = ufl.Measure("dx", domain=domain, subdomain_data=ct)

In [3]:
from dolfinx import fem, default_scalar_type
import numpy as np
import ufl
V = fem.functionspace(domain, ("Lagrange", 1))
u = ufl.TrialFunction(V)
w = ufl.TestFunction(V)

In [4]:
# Physics parameters 
K = ufl.as_tensor([[1.0, 0.0], [0.0, 0.0]])
diffusion_on = True
advection_on = False

In [5]:
# Source term
f = fem.Constant(domain, default_scalar_type(1.0))
x = ufl.SpatialCoordinate(domain)
# Boundary functions 
g_map = {1: x[0], 2: x[0], 3: x[1], 4:x[1]}


In [6]:
# Nitsche parameters
gamma = fem.Constant(domain, default_scalar_type(1.0))
penalty = fem.Constant(domain, default_scalar_type(8.0))
h = ufl.CellDiameter(domain)
n = ufl.FacetNormal(domain)

In [7]:
# Bilinear term
a = ufl.inner(K * ufl.grad(u), ufl.grad(w)) * ufl.dx
Kgradu_n = ufl.dot(-K * ufl.grad(u), n)
Kgradw_n = ufl.dot(-gamma * K * ufl.grad(w), n)
# Extra terms for bilinear form (LHS) 
a += Kgradu_n * w * ds 
a += Kgradw_n * u * ds

In [8]:
# Linear term
L = f * w * ufl.dx 
for tag, g in g_map.items():
    L += Kgradw_n * g * ds(tag)

In [9]:
# Penalty terms 
C = 5 
knorm = ufl.sqrt(ufl.inner(K, K))
# Penalty for bilinear form (LHS)
a += (C * knorm / h) * w * u * ds 
# Pennalty for linear form (RHS)
for tag, g in g_map.items():
    L += (C * knorm / h) * w * g * ds(tag)

In [10]:
from dolfinx.fem.petsc import LinearProblem 
from dolfinx.io import XDMFFile
problem = LinearProblem(
    a,
    L,
    bcs=[],
    petsc_options_prefix="basic_linear_problem",
    petsc_options={
        "ksp_type": "preonly",
        "pc_type": "lu",
    },
)
u_h = problem.solve()
u_h.name = "u"

outname = "solution"
with XDMFFILE(domain.comm, f"{outname}.xdmf", "w") as xdmf:
    xdmf.write_mesh(domain)
    xdmf.write_function(u_h)

TimeoutError: JIT compilation timed out, probably due to a failed previous compile. Try cleaning cache (e.g. remove /root/.cache/fenics/libffcx_forms_c0d4c821a31a05b3fe2ed821d81da11591e7d8da.c) or increase timeout option.